# DSFB Oil & Gas — Real-World Empirical Notebook

**Drift–Slew Fusion Bootstrap: Structural Residual Semantics for Upstream and Midstream Oil and Gas Systems**

---

## Positioning

DSFB does **not** compete with probabilistic anomaly detection, RTTM models,
Kalman filters, or ML classifiers.  It **augments** them.

Most production-monitoring pipelines produce residuals (observed minus expected)
and then immediately discard their temporal structure — compressing an entire
drift transient into a single alert threshold crossing.  DSFB applies a
deterministic structural grammar to those residuals *before* they are discarded,
converting raw residual sequences into labelled grammar episodes.  The episode
log is **human-readable and deterministic** and can be consumed by any downstream
probabilistic or ML stage.  Existing methods become *more* powerful, not redundant.

---

## Data Provenance & Academic Integrity

| Dataset | Licence | Steps |
|---------|---------|-------|
| Petrobras 3W v2.0.0 | CC BY 4.0 | 9 087 |
| Equinor Volve 15/9-F-15 WITSML | Equinor Volve Data Licence V1.0 | 5 326 |
| RPDBCS ESPset | MIT License | 6 032 |

> DSFB is a read-only observer.  It produces grammar annotations on residuals;
> it does **not** predict failures, detect anomalies in any ML sense, control
> systems, or issue alarms.  Fault labels are used *solely* as post-hoc
> annotation metadata — they were never provided to the DSFB engine during
> processing.

---

## What this notebook does

1. Installs Rust **from scratch** (rustup stable) if absent
2. Builds the `dsfb-oil-gas` crate with `cargo build --release`
3. Runs `cargo run --release --example export_grammar_traces` → per-step annotation CSVs
4. Generates ≥12 publication-quality figures from real data only
5. Compiles a PDF report and packages everything into a zip for download

**No synthetic or simulated data appear anywhere in this notebook.**

## 0a · Mathematical Definition

For one scalar channel, the executable Rust implementation computes the DSFB
state in four deterministic stages:

1. Residual: $r_k = observed_k - expected_k$
2. Drift: $\delta_k^{(w)} = \frac{1}{m_k} \sum_{j=\max(0, k-w+1)}^k r_j$, where $m_k = \min(w, k+1)$
3. Slew: $\sigma_k = \frac{r_k - r_{k-1}}{\Delta t_k}$, with $\sigma_0 = 0$
4. Envelope normalisation: each coordinate is mapped affinely into $[-1, 1]$

```text
r̃_k = (r_k     - (r_min     + r_max    ) / 2) / ((r_max     - r_min    ) / 2)
δ̃_k = (δ_k^(w) - (delta_min + delta_max) / 2) / ((delta_max - delta_min) / 2)
σ̃_k = (σ_k     - (sigma_min + sigma_max) / 2) / ((sigma_max - sigma_min) / 2)
```

Grammar precedence in `src/grammar.rs`:

```text
Compound > EnvViolation > SlewSpike > DriftAccum > BoundaryGrazing > Recovery > Nominal
```

Implementation details that matter for reproduction:

- Drift warm-up uses the partial-window mean `m_k = min(w, k+1)`.
- `Δt_k` falls back to `1.0` on the first sample or when timestamps do not strictly increase.
- Non-finite residuals emit `SensorFault` and do not update the drift ring buffer or slew state.

## 0b · Minimal Rust API Example

```rust
use dsfb_oil_gas::{
    aggregate_episodes, format_summary, summarise,
    AdmissibilityEnvelope, DeterministicDsfb, GrammarClassifier, ResidualSample,
};

fn main() {
    let env = AdmissibilityEnvelope::default_pipeline();
    let mut engine = DeterministicDsfb::with_window(
        env,
        GrammarClassifier::new(),
        10,
        "pipeline_flow_balance",
    );

    let samples = [
        ResidualSample::new(0.0, 101.2, 100.9, "pipeline_flow_balance"),
        ResidualSample::new(1.0, 101.6, 101.0, "pipeline_flow_balance"),
        ResidualSample::new(2.0, 106.8, 101.1, "pipeline_flow_balance"),
    ];

    for sample in &samples {
        engine.ingest_sample(sample);
    }

    let episodes = aggregate_episodes(engine.history());
    let summary = summarise("pipeline_flow_balance", engine.history(), &episodes);
    println!("{}", format_summary(&summary));
}
```

For built-in domain frames, call `engine.ingest(frame)` instead of constructing
`ResidualSample` manually.

## 0c · How To Run The Crate

From the crate root (`crates/dsfb-oil-gas`), the main entrypoints are:

```bash
# Demo binary: synthetic pipeline CSV
cargo run -- data/pipeline_synthetic.csv

# Real-data metric summaries used in the paper discussion
cargo run --example metrics_3w
cargo run --example metrics_volve
cargo run --example metrics_esp

# Real-data integration tests
cargo test --test real_data_3w
cargo test --test real_data_volve
cargo test --test real_data_esp

# Export per-step grammar traces used by the figure pipeline
cargo run --release --example export_grammar_traces

# Generate the full figure set
cargo run --release -- generate-figures
```

## 0 · Environment Setup

Detects Google Colab vs local Jupyter.  Installs Rust if absent (≈ 60 s).

In [ ]:
import sys, os, pathlib, subprocess, shutil

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print(f"Runtime: {'Google Colab' if IN_COLAB else 'local Jupyter / VS Code'}")

CARGO = shutil.which('cargo')
if CARGO is None:
    print('Rust not found — installing via rustup (~60 s)...')
    subprocess.run(
        'curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs'
        ' | sh -s -- -y --default-toolchain stable --no-modify-path',
        shell=True, check=True
    )
    CARGO_HOME = pathlib.Path.home() / '.cargo'
    os.environ['PATH'] = str(CARGO_HOME / 'bin') + ':' + os.environ.get('PATH', '')
    CARGO = str(CARGO_HOME / 'bin' / 'cargo')
else:
    cargo_bin = str(pathlib.Path(CARGO).parent)
    if cargo_bin not in os.environ.get('PATH', ''):
        os.environ['PATH'] = cargo_bin + ':' + os.environ.get('PATH', '')

print(subprocess.check_output([CARGO, '--version']).decode().strip())
print(subprocess.check_output(['rustc', '--version']).decode().strip())

In [ ]:
import importlib.util

for pkg in ['numpy', 'pandas', 'matplotlib', 'scipy']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

print(f'numpy {np.__version__}  pandas {pd.__version__}  matplotlib {matplotlib.__version__}')

## 1 · Repository & Data Paths

**Local:** notebook lives at `<crate>/notebook/dsfb_oil_gas.ipynb`; crate root is at `../../crates/dsfb-oil-gas`.

**Colab:** uses a Google Drive checkout if `DRIVE_REPO_PATH` exists; otherwise it clones the repo
into `/content/dsfb` automatically before running any Rust commands.

In [ ]:
REPO_URL = 'https://github.com/infinityabundance/dsfb.git'
DRIVE_REPO_PATH = pathlib.Path('/content/drive/MyDrive/dsfb')  # optional persisted repo root
CLONE_ROOT = pathlib.Path('/content/dsfb')

def _resolve_layout(candidate):
    candidate = pathlib.Path(candidate).expanduser().resolve()
    crate = candidate / 'crates' / 'dsfb-oil-gas'
    if (crate / 'Cargo.toml').exists():
        return candidate, crate
    if (candidate / 'Cargo.toml').exists() and (candidate / 'src').exists():
        return candidate.parent, candidate
    return None

if IN_COLAB:
    resolved = None
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        if DRIVE_REPO_PATH.exists():
            resolved = _resolve_layout(DRIVE_REPO_PATH)
    except Exception as exc:
        print(f'Drive issue: {exc}')

    if resolved is None:
        for candidate in [CLONE_ROOT, pathlib.Path('/content/dsfb-oil-gas'), pathlib.Path.cwd()]:
            resolved = _resolve_layout(candidate)
            if resolved is not None:
                break

    if resolved is None:
        clone_target = CLONE_ROOT
        if clone_target.exists() and not (clone_target / '.git').exists():
            clone_target = pathlib.Path('/content/dsfb-colab')
        print(f'Cloning {REPO_URL} -> {clone_target} ...')
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(clone_target)], check=True)
        resolved = _resolve_layout(clone_target)

    if resolved is None:
        raise FileNotFoundError('Failed to locate crates/dsfb-oil-gas after Colab bootstrap')

    WORKSPACE_ROOT, CRATE_ROOT = resolved
    OUTPUT_DIR = WORKSPACE_ROOT / 'colab_output'
else:
    try:
        NB_DIR = pathlib.Path(__file__).parent.resolve()
    except NameError:
        NB_DIR = pathlib.Path.cwd().resolve()
    resolved = None
    for candidate in [NB_DIR, NB_DIR.parent, NB_DIR.parent.parent, NB_DIR.parent.parent.parent]:
        resolved = _resolve_layout(candidate)
        if resolved is not None:
            break
    if resolved is None:
        raise FileNotFoundError('Could not locate crates/dsfb-oil-gas from the notebook directory')
    WORKSPACE_ROOT, CRATE_ROOT = resolved
    OUTPUT_DIR = CRATE_ROOT

REPO        = CRATE_ROOT          # cargo commands run from here
DATA_DIR    = CRATE_ROOT / 'data'
TRACE_DIR   = CRATE_ROOT / 'figures' / 'trace_data'
FIGURES_DIR = OUTPUT_DIR / 'figures'
COLAB_OUT   = OUTPUT_DIR / 'colab' / 'output'
COLAB_OUT.mkdir(parents=True, exist_ok=True)
TRACE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Crate root : {CRATE_ROOT}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'Colab out  : {COLAB_OUT}')

In [ ]:
REAL_FILES = {
    'oilwell_real.csv':  ('Petrobras 3W',    'CC BY 4.0',   9300),  # raw CSV rows; Rust filters to 9087 trace steps
    'drilling_real.csv': ('Equinor Volve',   'VDL V1.0',    5326),
    'rotating_real.csv': ('RPDBCS ESPset',   'MIT',         6032),
}

all_present = True
print(f"{'File':<25} {'Dataset':<20} {'Licence':<12} {'Rows':>6}")
print('-' * 68)
for fn, (name, lic, expected) in REAL_FILES.items():
    path = DATA_DIR / fn
    if path.exists():
        rows = sum(1 for _ in open(path)) - 1
        print(f'{fn:<25} {name:<20} {lic:<12} {rows:>6}')
    else:
        print(f'{fn:<25} MISSING (see README.md for download instructions)')
        all_present = False

if all_present:
    print('\nAll datasets present.')
else:
    print('\nMissing files — see README.md')

## 2 · Build the Rust Crate from Scratch

`cargo build --release` compiles the full crate.  First run downloads all
dependencies (≈ 90 s in Colab; ≈ 5 s cached locally).

In [ ]:
import time

print('cargo build --release --example export_grammar_traces')
print('(first run may take 60-120 s in Colab)')
t0 = time.time()

result = subprocess.run(
    [CARGO, 'build', '--release', '--example', 'export_grammar_traces'],
    cwd=str(REPO), capture_output=True, text=True
)
elapsed = time.time() - t0

if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError('cargo build failed')

for line in result.stderr.splitlines():
    if 'Finished' in line:
        print(line.strip())
        break

print(f'Build complete in {elapsed:.1f} s')

In [ ]:
result = subprocess.run(
    [CARGO, 'test', '--quiet'],
    cwd=str(REPO), capture_output=True, text=True
)
for line in result.stdout.splitlines():
    if 'test result' in line:
        print(line)
if result.returncode != 0:
    raise RuntimeError('cargo test failed')
print('All tests pass.')

## 2b · Kani Formal Verification

Kani is a Rust model checker that proves properties over **all possible inputs**
using bounded model checking — not sampling.

14 harnesses in `src/kani_proofs.rs` cover five provable code-level claims:

| Proof | Claim | Property |
|-------|-------|----------|
| `proof_coord_classify_deterministic` | Determinism | Same val+band → same CoordClass |
| `proof_grammar_classify_deterministic` | Determinism | Two fresh classifiers → same output |
| `proof_slew_first_push_is_zero` | Determinism | First push always 0.0 |
| `proof_grammar_classify_does_not_mutate_triple` | Non-interference | classify() never mutates its triple |
| `proof_slew_second_push_is_finite_difference` | Pipeline | σ = (r₂−r₁)/dt exactly |
| `proof_slew_zero_dt_returns_zero` | Pipeline | dt=0 guard, no divide-by-zero |
| `proof_envelope_origin_is_interior` | Envelope | (0,0,0) is Interior |
| `proof_coord_value_beyond_boundary_is_outside` | Envelope | \|val\|>1 → Outside |
| `proof_coord_strictly_interior_region` | Envelope | \|val\|<(1−band) → Interior |
| `proof_grammar_compound_precedence` | Grammar | δ-violated ∧ σ-violated → Compound |
| `proof_grammar_envviolation_precedence` | Grammar | r-violated ∧ ¬Compound → EnvViolation |
| `proof_grammar_sensor_fault_on_nonfinite_r` | Grammar | NaN r → SensorFault |
| `proof_grammar_all_interior_from_nominal_gives_nominal` | Grammar | all-interior → Nominal |
| `proof_grammar_classify_never_panics` | Grammar | no symbolic input panics |

In [ ]:
import shutil

KANI = shutil.which('kani') or shutil.which('cargo-kani')

if KANI is None:
    print('Kani not found.  To install and run the 14 formal proofs:')
    print()
    print('  cargo install --locked kani-verifier')
    print('  cargo kani setup')
    print('  cd', str(REPO))
    print('  cargo kani                                     # all 14 proofs')
    print('  cargo kani --harness proof_grammar_compound_precedence  # single proof')
    print()
    print('Kani proofs are in src/kani_proofs.rs.')
    print('They are compiled ONLY by `cargo kani`; they do not affect normal builds.')
else:
    print('Kani found — running all formal verification harnesses...')
    result = subprocess.run(
        ['cargo', 'kani', '--output-format', 'terse'],
        cwd=str(REPO), capture_output=True, text=True, timeout=600,
    )
    print(result.stdout[-6000:] if len(result.stdout) > 6000 else result.stdout)
    if result.returncode == 0:
        print('\nAll Kani proofs: VERIFICATION SUCCESSFUL')
    else:
        print('\nOne or more proofs failed:', result.stderr[-2000:])

In [ ]:
print('Running DSFB grammar engine on all real datasets...')
t0 = time.time()

result = subprocess.run(
    [CARGO, 'run', '--release', '--example', 'export_grammar_traces'],
    cwd=str(REPO), capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError('export_grammar_traces failed')

print(result.stdout)
print(f'Completed in {time.time()-t0:.1f} s')

for fn in ['real_3w_trace.csv', 'real_volve_trace.csv', 'real_esp_trace.csv']:
    p = TRACE_DIR / fn
    rows = sum(1 for _ in open(p)) - 1
    print(f'  {fn:<30} {rows:>6} rows')

## 3 · Load Trace Data & Summary Statistics

In [ ]:
d3w    = pd.read_csv(TRACE_DIR / 'real_3w_trace.csv')
dvolve = pd.read_csv(TRACE_DIR / 'real_volve_trace.csv')
desp   = pd.read_csv(TRACE_DIR / 'real_esp_trace.csv')

def _load_env(name):
    df = pd.read_csv(TRACE_DIR / f'env_{name}.csv', index_col=0)
    return df['value'].to_dict()

e3w    = _load_env('3w')
evolve = _load_env('volve')
eesp   = _load_env('esp')

TOK_ORDER = ['Nominal','DriftAccum','SlewSpike','EnvViolation',
             'BoundaryGrazing','Recovery','Compound']

def _ncr(df):
    K = len(df); E = (df['token'] != 'Nominal').sum()
    return K / E if E > 0 else float('inf')

print(f"{'Dataset':<22} {'Steps':>7} {'%Nominal':>9} {'%EnvViol':>9} {'NCR':>7}")
print('=' * 60)
for label, df in [('Petrobras 3W', d3w), ('Equinor Volve', dvolve), ('RPDBCS ESP', desp)]:
    nom = (df['token'] == 'Nominal').mean() * 100
    ev  = (df['token'] == 'EnvViolation').mean() * 100
    print(f'{label:<22} {len(df):>7,} {nom:>8.1f}% {ev:>8.1f}% {_ncr(df):>7.1f}')
print('=' * 60)
print('NCR = steps / non-Nominal (higher => more compression)')

In [ ]:
plt.rcParams.update({
    'font.family': 'serif', 'font.size': 9, 'axes.titlesize': 9,
    'axes.labelsize': 8.5, 'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5,
    'legend.fontsize': 7.5, 'figure.dpi': 150, 'axes.linewidth': 0.6,
    'grid.linewidth': 0.35, 'grid.alpha': 0.45, 'lines.linewidth': 0.8,
    'pdf.fonttype': 42,
})

TC = {
    'Nominal': '#2166ac', 'DriftAccum': '#f4a582', 'SlewSpike': '#d62028',
    'EnvViolation': '#762a83', 'BoundaryGrazing': '#74c476',
    'Recovery': '#fdae61', 'Compound': '#111111',
}

def _legend_patches():
    return [mpatches.Patch(color=TC[t], label=t) for t in TOK_ORDER]

def _shade_tokens(ax, idx, tokens, alpha=0.18):
    toks = list(tokens)
    if not toks: return
    prev = toks[0]; start = idx[0]
    for x, tok in zip(idx, toks):
        if tok != prev:
            ax.axvspan(start, x, color=TC.get(prev,'#ccc'), alpha=alpha, linewidth=0)
            prev = tok; start = x
    ax.axvspan(start, idx[-1], color=TC.get(prev,'#ccc'), alpha=alpha, linewidth=0)

def _envelope_box(ax, xlo=-1, xhi=1, ylo=-1, yhi=1):
    rect = mpatches.Rectangle((xlo,ylo), xhi-xlo, yhi-ylo,
                               fill=False, edgecolor='black',
                               linewidth=0.9, linestyle='--', zorder=5)
    ax.add_patch(rect)

SAVED_FIGS = []

def _save(fig, name):
    pdf_path = COLAB_OUT / f'{name}.pdf'
    png_path = COLAB_OUT / f'{name}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, bbox_inches='tight', dpi=150)
    SAVED_FIGS.append(pdf_path)
    print(f'  saved: {pdf_path.name}')
    plt.show()
    plt.close(fig)

print('Plot configuration and colour palette ready.')

## 4 · Petrobras 3W Results

Dataset: Petrobras 3W v2.0.0 — CC BY 4.0 — https://github.com/petrobras/3W

Real `WELL-*` instances only (no `SIMULATED_*` or `DRAWN_*` files).
Primary channel: P-MON-CKP.  Baseline: 30-minute causal rolling median.

In [ ]:
# Fig A1: Full P-MON-CKP residual trace, grammar-token shaded
fig, axes = plt.subplots(2, 1, figsize=(13, 5.2), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
ax, ax2 = axes

idx = d3w['step_idx'].values
res = d3w['residual_pa'].values / 1e3

_shade_tokens(ax, idx, d3w['token'])
for tok in TOK_ORDER:
    mask = (d3w['token'] == tok).values
    if mask.any():
        ax.scatter(idx[mask], res[mask], s=0.5, color=TC[tok], linewidths=0, rasterized=True)

ax.axhline(float(e3w['r_max'])/1e3, color='k', lw=0.8, ls='--')
ax.axhline(float(e3w['r_min'])/1e3, color='k', lw=0.8, ls='--')
ax.axhline(0, color='#666', lw=0.5, ls=':')
ax.set_ylabel('Residual (kPa)')
ax.set_title(
    'Petrobras 3W v2.0.0 — P-MON-CKP residual coloured by DSFB grammar token\n'
    '9 087 steps  |  12 real WELL-* instances  |  CC BY 4.0  |  '
    'Fault labels withheld from engine',
    loc='left', fontsize=8.5)
ax.legend(handles=_legend_patches(), ncol=4, fontsize=6.5, loc='upper right', framealpha=0.85)
ax.grid(True, axis='y')

eq = d3w['event_class'].dropna()
if len(eq) > 0:
    for ec in sorted(eq.unique()):
        mask = (d3w['event_class'] == ec).values
        ax2.scatter(idx[mask], [ec]*mask.sum(), s=0.5, color='#333', linewidths=0, rasterized=True)
ax2.set_ylabel('3W class', fontsize=7)
ax2.set_xlabel('Step index')
ax2.grid(True, axis='y')

fig.tight_layout(h_pad=0.4)
_save(fig, 'figA1_3w_residual_trace')

In [ ]:
# Fig A2: Per-episode token distribution — stacked horizontal bar
wells = (d3w.groupby('episode_name')['token']
           .value_counts(normalize=True)
           .unstack(fill_value=0)
           .reindex(columns=TOK_ORDER, fill_value=0)
           .sort_values('Nominal', ascending=True))

fig, ax = plt.subplots(figsize=(9, max(3, 0.5*len(wells)+1.5)))
lefts = np.zeros(len(wells))
for tok in TOK_ORDER:
    vals = wells[tok].values * 100
    ax.barh(wells.index, vals, left=lefts, color=TC[tok], height=0.72, linewidth=0)
    for j, (v, l) in enumerate(zip(vals, lefts)):
        if v >= 5:
            ax.text(l+v/2, j, f'{v:.0f}', ha='center', va='center',
                    fontsize=5.5, color='white' if v>10 else '#333')
    lefts += vals

ax.set_xlim(0, 100)
ax.set_xlabel('Fraction of steps (%)')
ax.set_title(
    'Petrobras 3W — DSFB token fraction by well episode (sorted by Nominal%)\n'
    'CC BY 4.0  |  No fault labels used during grammar processing',
    loc='left', fontsize=8.5)
ax.legend(handles=_legend_patches(), loc='lower right', fontsize=6, ncol=2, framealpha=0.85)
ax.grid(True, axis='x', lw=0.35)
fig.tight_layout()
_save(fig, 'figA2_3w_per_episode_tokens')

In [ ]:
# Fig A3: Phase portrait (r_norm, delta_norm) coloured by token
fig, ax = plt.subplots(figsize=(4.8, 4.4))
for tok in reversed(TOK_ORDER):
    mask = (d3w['token'] == tok).values
    if mask.any():
        sub = d3w[mask]
        ax.scatter(sub['r_norm'], sub['delta_norm'], s=0.7,
                   color=TC[tok], linewidths=0, rasterized=True, alpha=0.75)

_envelope_box(ax, -1, 1, -1, 1)
ax.axhline(0, color='#aaa', lw=0.4, ls=':'); ax.axvline(0, color='#aaa', lw=0.4, ls=':')
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_xlabel('r_norm (normalised residual)')
ax.set_ylabel('delta_norm (normalised drift accumulator)')
ax.set_title(
    '3W — normalised phase portrait by DSFB grammar token\n'
    'Dashed box = admissibility boundary (+-1 in both dimensions)',
    loc='left', fontsize=8.5)
ax.legend(handles=_legend_patches(), fontsize=6, markerscale=3, loc='upper right', framealpha=0.85)
ax.set_aspect('equal', 'box')
ax.grid(True, lw=0.35, alpha=0.45)
fig.tight_layout()
_save(fig, 'figA3_3w_phase_portrait')

## 5 · Equinor Volve Results

Dataset: Equinor Volve 15/9-F-15 WITSML logs — Equinor Volve Data Licence V1.0  
https://data.equinor.com/dataset/Volve  *(requires Equinor account)*

Primary channel: surface torque TQA (kNm), depth-indexed at 0.5 m intervals.  
Drill interval: 1 200 – 4 065 m.  Baseline: causal 10-m rolling median.

In [ ]:
# Fig B1: TQA residual vs depth, grammar-token shaded
fig, ax = plt.subplots(figsize=(13, 4.0))
depth = dvolve['depth_m'].values
res_v = dvolve['residual_knm'].values

_shade_tokens(ax, depth, dvolve['token'])
for tok in TOK_ORDER:
    mask = (dvolve['token'] == tok).values
    if mask.any():
        ax.scatter(depth[mask], res_v[mask], s=0.8, color=TC[tok], linewidths=0, rasterized=True)

ax.axhline(float(evolve['r_max']), color='k', lw=0.8, ls='--')
ax.axhline(float(evolve['r_min']), color='k', lw=0.8, ls='--')
ax.axhline(0, color='#666', lw=0.5, ls=':')
ax.set_xlabel('Measured depth (m)')
ax.set_ylabel('TQA residual (kNm)')
ax.set_title(
    'Equinor Volve 15/9-F-15 — TQA residual coloured by DSFB grammar token\n'
    '5 326 depth-steps at 0.5 m resolution  |  Volve DLV1.0  |  '
    'No formation labels used during grammar processing',
    loc='left', fontsize=8.5)
ax.legend(handles=_legend_patches(), ncol=4, fontsize=6.5, loc='upper right', framealpha=0.85)
ax.grid(True, axis='y')
fig.tight_layout()
_save(fig, 'figB1_volve_tqa_trace')

In [ ]:
# Fig B2: drift and slew channels vs depth
depth = dvolve['depth_m'].values
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 5.6), sharex=True)

for tok in TOK_ORDER:
    mask = (dvolve['token'] == tok).values
    if mask.any():
        ax1.scatter(depth[mask], dvolve['drift_knm'].values[mask],
                    s=0.6, color=TC[tok], linewidths=0, rasterized=True)
        ax2.scatter(depth[mask], dvolve['slew_knm'].values[mask],
                    s=0.6, color=TC[tok], linewidths=0, rasterized=True)

for ax, key_hi, key_lo, ylabel in [
    (ax1, 'delta_max', 'delta_min', 'drift accumulator (kNm)'),
    (ax2, 'sigma_max', 'sigma_min', 'slew rate (kNm)'),
]:
    ax.axhline(float(evolve[key_hi]), color='k', lw=0.8, ls='--')
    ax.axhline(float(evolve[key_lo]), color='k', lw=0.8, ls='--')
    ax.axhline(0, color='#666', lw=0.4, ls=':')
    ax.set_ylabel(ylabel, fontsize=7.5)
    ax.grid(True, axis='y')
    ax.legend(handles=_legend_patches(), ncol=7, fontsize=5, loc='upper right', framealpha=0.85)

ax1.set_title(
    'Volve — drift accumulator and slew rate vs measured depth\n'
    'High-|drift| near 2400 m: slow torque buildup (DriftAccum) — '
    'geometrically distinct from instantaneous stick-slip (SlewSpike)',
    loc='left', fontsize=8.5)
ax2.set_xlabel('Measured depth (m)')
fig.tight_layout(h_pad=0.4)
_save(fig, 'figB2_volve_drift_slew')

## 6 · RPDBCS ESP Results

Dataset: RPDBCS ESPset — MIT License — https://github.com/Rogerio-Chaves/RPDBCS

6 032 vibration snapshots from 11 ESP units; 5 fault classes (Normal, Unbalance,
Faulty sensor, Rubbing, Misalignment).  Primary DSFB channel: broadband RMS (98–102 Hz).

> **Key caveat:** ESPset is a snapshot classification corpus, not a continuous
> telemetry stream.  Unit-boundary crossings introduce residual resets that inflate
> episode count and deflate NCR relative to a field installation.
> **No detection or predictive claim is made.**

In [ ]:
# Fig C1: Per-unit EnvViolation rate vs true fault rate
rows = []
for eid, grp in desp.groupby('esp_id'):
    n = len(grp)
    rows.append({'esp_id': int(eid),
                 'fault_rate': (grp['label'] != 'Normal').sum() / n,
                 'envv_rate':  (grp['token'] == 'EnvViolation').sum() / n})
dfunits = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(5.0, 4.4))
ax.scatter(dfunits['fault_rate']*100, dfunits['envv_rate']*100,
           s=60, color='#2166ac', edgecolors='white', linewidths=0.5, zorder=4)
for _, r in dfunits.iterrows():
    ax.text(r['fault_rate']*100+0.6, r['envv_rate']*100, str(r['esp_id']), fontsize=6.5)

hi = max(dfunits['fault_rate'].max(), dfunits['envv_rate'].max())*100 + 8
ax.plot([0, hi], [0, hi], 'k--', lw=0.9, label='y=x (perfect alignment)')
ax.set_xlabel('True fault-step fraction (%, ground-truth label)')
ax.set_ylabel('DSFB EnvViolation token fraction (%)')
ax.set_title(
    'RPDBCS ESP — per-unit EnvViolation rate vs true fault rate\n'
    'POST-HOC ALIGNMENT CHECK ONLY — not a detection metric\n'
    'DSFB received no fault labels during processing',
    loc='left', fontsize=8)
ax.legend(fontsize=7, framealpha=0.85)
ax.grid(True, lw=0.35, alpha=0.45)
ax.annotate('11 ESP units  |  RPDBCS ESPset  |  MIT Licence\n'
            'Agreement is statistically incidental; causal interpretation unwarranted.',
            xy=(0.02, 0.97), xycoords='axes fraction', va='top', fontsize=6, color='#555')
fig.tight_layout()
_save(fig, 'figC1_esp_envviol_vs_faultrate')

In [ ]:
# Fig C2: Broadband RMS distribution by fault label
labels_order = sorted(desp['label'].dropna().unique().tolist())
data_bp = [desp[desp['label']==lbl]['rms_broadband'].dropna().values for lbl in labels_order]
data_bp = [d[np.isfinite(d)] for d in data_bp]
baseline_med = desp['baseline_rms'].median()
env_hi = baseline_med + float(eesp['r_max'])

fig, ax = plt.subplots(figsize=(6.5, 3.8))
bp = ax.boxplot(data_bp, vert=True, patch_artist=True, notch=True,
                widths=0.55, showfliers=False,
                medianprops=dict(color='white', lw=1.2),
                whiskerprops=dict(lw=0.7), capprops=dict(lw=0.7))
clrs = plt.cm.tab10(np.linspace(0, 0.9, len(labels_order)))
for patch, c in zip(bp['boxes'], clrs):
    patch.set_facecolor(c)

ax.axhline(env_hi, color='#762a83', ls='--', lw=0.9,
           label=f'Baseline median + r_max ({float(eesp["r_max"]):.4f})')
ax.set_xticks(range(1, len(labels_order)+1))
ax.set_xticklabels(labels_order, rotation=18, ha='right', fontsize=7.5)
ax.set_ylabel('RMS broadband (normalised)')
ax.set_title(
    'RPDBCS ESP — broadband RMS by ground-truth fault label  (notched box plot)\n'
    'Labels provided post-hoc for display only — not seen by DSFB during processing',
    loc='left', fontsize=8.5)
ax.legend(fontsize=7, framealpha=0.85)
ax.grid(True, axis='y', lw=0.35, alpha=0.45)
fig.tight_layout()
_save(fig, 'figC2_esp_rms_by_class')

## 7 · DSFB as Augmentation — Residual Structure Preservation

A threshold-crossing alarm produces one bit of information.  DSFB retains the
full temporal grammar of the event — onset, plateau, recovery.  The next figure
shows both representations side by side on the same 3W episode.

In [ ]:
# Fig E1: DSFB augmentation illustration — grammar vs binary alert
ep_sizes  = d3w.groupby('episode_name').size().sort_values(ascending=False)
largest_ep = ep_sizes.index[0]
sub = d3w[d3w['episode_name'] == largest_ep].reset_index(drop=True)

fig, axes = plt.subplots(3, 1, figsize=(12, 7.4), sharex=True,
                         gridspec_kw={'height_ratios': [2.5, 1, 1]})
ax_res, ax_naive, ax_tok = axes

idx  = sub['step_idx'].values
res  = sub['residual_pa'].values / 1e3
emax = float(e3w['r_max']) / 1e3

# Panel 1: DSFB-annotated
_shade_tokens(ax_res, idx, sub['token'])
for tok in TOK_ORDER:
    mask = (sub['token'] == tok).values
    if mask.any():
        ax_res.scatter(idx[mask], res[mask], s=1.0, color=TC[tok], linewidths=0, rasterized=True)
ax_res.axhline(emax, color='k', lw=0.8, ls='--')
ax_res.axhline(-emax, color='k', lw=0.8, ls='--')
ax_res.axhline(0, color='#666', lw=0.5, ls=':')
ax_res.set_ylabel('Residual (kPa)')
ax_res.set_title(
    f'3W episode "{largest_ep}" ({len(sub):,} steps)\n'
    'Panel 1 (DSFB): grammar-annotated residual — temporal structure preserved for downstream methods',
    loc='left', fontsize=8.5)
ax_res.legend(handles=_legend_patches(), ncol=4, fontsize=6, loc='lower right', framealpha=0.85)
ax_res.grid(True, axis='y')

# Panel 2: naive threshold
alert = (np.abs(res) > emax).astype(int)
ax_naive.fill_between(idx, 0, alert, step='mid', color='#d62028', alpha=0.7)
ax_naive.set_ylabel('Alert (0/1)', fontsize=7)
ax_naive.set_ylim(-0.05, 1.35)
ax_naive.set_yticks([0, 1])
ax_naive.grid(True, axis='y')
ax_naive.set_title('Panel 2 (naive threshold): binary alert only — '
                   'onset ramp, plateau, and recovery are discarded', loc='left', fontsize=8)
n_al = alert.sum()
ax_naive.annotate(f'{n_al}/{len(alert)} steps breach threshold '
                  f'({n_al/len(alert)*100:.1f}%) — temporal structure completely lost',
                  xy=(0.02, 0.78), xycoords='axes fraction', fontsize=6.5, color='#333')

# Panel 3: DSFB token strip
for i in range(len(idx)-1):
    ax_tok.fill_between([idx[i], idx[i+1]], 0, 1, step='mid',
                        color=TC[sub['token'].iloc[i]], alpha=0.88, linewidth=0)
ax_tok.set_ylabel('Token', fontsize=7)
ax_tok.set_yticks([])
ax_tok.set_xlabel('Step index')
ax_tok.set_title('Panel 3 (DSFB token strip): onset -> EnvViolation -> Recovery '
                 'is a machine-readable event signature consumable by downstream ML',
                 loc='left', fontsize=8)
for tok in ['EnvViolation', 'DriftAccum', 'Recovery']:
    mask = (sub['token'] == tok).values
    if mask.any():
        cx = sub['step_idx'].values[mask].mean()
        ax_tok.text(cx, 0.5, tok, ha='center', va='center',
                    fontsize=5.5, color='white', fontweight='bold')

fig.tight_layout(h_pad=0.5)
_save(fig, 'figE1_augmentation_illustration')

## 8 · Cross-Dataset Analysis

In [ ]:
# Fig D1: Cross-dataset token heatmap
dsets = {'3W (real)': d3w, 'Volve (real)': dvolve, 'ESP (real)': desp}
matrix = np.zeros((len(dsets), len(TOK_ORDER)))
for i, (nm, df) in enumerate(dsets.items()):
    vc = df['token'].value_counts(normalize=True) * 100
    for j, tok in enumerate(TOK_ORDER):
        matrix[i, j] = vc.get(tok, 0.0)

fig, ax = plt.subplots(figsize=(9.5, 2.6))
im = ax.imshow(matrix, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100)
ax.set_xticks(range(len(TOK_ORDER)))
ax.set_xticklabels(TOK_ORDER, rotation=22, ha='right', fontsize=8)
ax.set_yticks(range(len(dsets)))
ax.set_yticklabels(list(dsets.keys()), fontsize=8.5)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        v = matrix[i, j]
        ax.text(j, i, f'{v:.1f}', ha='center', va='center',
                fontsize=7.5, color='black' if v < 65 else 'white')
plt.colorbar(im, ax=ax, label='% of steps', shrink=0.9)
ax.set_title('Cross-dataset DSFB token distribution (% of steps) — real data only',
             loc='left', fontsize=8.5)
fig.tight_layout()
_save(fig, 'figD1_cross_token_heatmap')

In [ ]:
# Fig D2: NCR bar chart — real datasets only (this notebook contains no synthetic data)
# NCR = K / non-Nominal steps.  Higher -> grammar condenses more steps per event.
# The wide spread across datasets reflects different physical domains and corpus
# structures, not differences in DSFB calibration quality.
def _ncr_val(df):
    K = len(df); E = (df['token'] != 'Nominal').sum()
    return K / E if E > 0 else float('nan')

labels = ['3W\n(real)', 'Volve\n(real)', 'ESP\n(real*)']
ncr_vals = [_ncr_val(d3w), _ncr_val(dvolve), _ncr_val(desp)]
colors   = ['#2166ac', '#4dac26', '#762a83']

fig, ax = plt.subplots(figsize=(5.5, 3.8))
for i, (lbl, val, col) in enumerate(zip(labels, ncr_vals, colors)):
    ax.bar(i, val, color=col, linewidth=0.5, edgecolor='white')
    ax.text(i, val+0.05, f'{val:.1f}', ha='center', va='bottom', fontsize=8.5,
            fontweight='bold')

ax.set_xticks(range(3))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('NCR = K / non-Nominal steps')
ax.set_title(
    'Noise compression ratio by dataset — real data only\n'
    'NCR varies across domains: Volve deepest (lowest event density),'
    ' ESP lowest (snapshot-corpus unit resets)',
    loc='left', fontsize=8.5)
ax.annotate(
    '* ESP NCR (1.8) deflated by snapshot-corpus unit-boundary resets; field NCR expected higher.\n'
    '  3W and Volve are continuous-stream datasets; ESP is a classification snapshot corpus.\n'
  '  NCR is a descriptor, not a quality or performance metric.',
    xy=(0.01, -0.34), xycoords='axes fraction', fontsize=6.5, color='#555')
ax.set_ylim(0, max(ncr_vals)*1.3 if max(ncr_vals) > 0 else 10)
ax.grid(True, axis='y', lw=0.35, alpha=0.45)
fig.tight_layout()
_save(fig, 'figD2_cross_ncr_bar')

In [ ]:
# Fig D3: Envelope utilisation
dsets2 = {'3W\n(real)': d3w, 'Volve\n(real)': dvolve, 'ESP\n(real)': desp}
dims    = ['|r~|>1', '|delta~|>1', '|sigma~|>1']
cols_k  = ['r_norm', 'delta_norm', 'sigma_norm']
bar_clr = ['#2166ac', '#d62028', '#f4a582']
x       = np.arange(len(dsets2))
wid     = 0.24

fig, ax = plt.subplots(figsize=(5.8, 3.8))
for k, (dim, bc) in enumerate(zip(dims, bar_clr)):
    vals = [(df[cols_k[k]].abs() > 1).mean()*100 for df in dsets2.values()]
    offset = (k - 1) * wid
    bars = ax.bar(x+offset, vals, width=wid*0.9, color=bc, label=dim, linewidth=0.4)
    for bar, v in zip(bars, vals):
        if v > 1:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                    f'{v:.1f}%', ha='center', va='bottom', fontsize=6.2)

ax.set_xticks(x)
ax.set_xticklabels(list(dsets2.keys()), fontsize=8.5)
ax.set_ylabel('Steps with |.|>1 (%)')
ax.set_title('Envelope utilisation — fraction of steps exceeding +-1 normalised bound',
             loc='left', fontsize=8.5)
ax.legend(fontsize=7.5, framealpha=0.85)
ax.grid(True, axis='y', lw=0.35, alpha=0.45)
fig.tight_layout()
_save(fig, 'figD3_cross_envelope_utilisation')

In [ ]:
# Fig D4: Cross-dataset episode-length violin
def _episode_lens(df):
    toks = df['token'].values; eps = []; cur = 0
    for tok in toks:
        if tok != 'Nominal': cur += 1
        elif cur > 0: eps.append(cur); cur = 0
    if cur > 0: eps.append(cur)
    return np.array(eps, dtype=float) if eps else np.array([1.0])

data_v  = [_episode_lens(d3w), _episode_lens(dvolve), _episode_lens(desp)]
vcols   = ['#2166ac', '#4dac26', '#762a83']
labels_v = ['3W (real)', 'Volve (real)', 'ESP (real)']

fig, ax = plt.subplots(figsize=(5.2, 4.0))
parts = ax.violinplot([np.log1p(v) for v in data_v],
                      positions=[1,2,3], widths=0.65,
                      showmedians=True, showextrema=True)
for pc, col in zip(parts['bodies'], vcols):
    pc.set_facecolor(col); pc.set_alpha(0.65)

ax.set_xticks([1,2,3])
ax.set_xticklabels(labels_v, fontsize=8.5)
yticks = [0,1,2,3,4,5]
ax.set_yticks(yticks)
ax.set_yticklabels([str(int(np.expm1(y))) if y>0 else '0' for y in yticks])
ax.set_ylabel('Episode length (steps, log-transformed axis)')
ax.set_title('Cross-dataset episode lengths — violin plot (log-transformed)\n'
             'Episodes = contiguous non-Nominal grammar token runs',
             loc='left', fontsize=8.5)
ax.grid(True, axis='y', lw=0.35, alpha=0.45)
fig.tight_layout()
_save(fig, 'figD4_cross_episode_violin')

## 9 · Summary Statistics

In [ ]:
print('DSFB Real-Dataset Token Distribution')
print('=' * 100)
hdr = f"{'Dataset':<28} {'K':>6} {'Nom%':>7} {'DA%':>7} {'SS%':>6} {'EV%':>6} {'BG%':>6} {'RC%':>6} {'NCR':>6}"
print(hdr)
print('-' * 100)
for label, df in [('Petrobras 3W  (CC BY 4.0)', d3w),
                  ('Equinor Volve (VDL V1.0)',   dvolve),
                  ('RPDBCS ESP    (MIT)',          desp)]:
    K  = len(df)
    vc = df['token'].value_counts(normalize=True) * 100
    p  = lambda t: vc.get(t, 0.0)
    ncr = _ncr_val(df)
    print(f"{label:<28} {K:>6,} {p('Nominal'):>6.1f}% {p('DriftAccum'):>6.1f}% "
          f"{p('SlewSpike'):>5.1f}% {p('EnvViolation'):>5.1f}% "
          f"{p('BoundaryGrazing'):>5.1f}% {p('Recovery'):>5.1f}% {ncr:>6.1f}")
print('=' * 100)
print()
print('Scope boundaries:')
print('  1. Single-channel evaluations only.')
print('  2. Envelope params calibrated on same data used for evaluation.')
print('  3. No fault detection, prediction, or anomaly detection claim.')
print('  4. ESP NCR reflects snapshot corpus structure, not field NCR.')
print('  5. Multi-channel / cross-asset generalisation: NOT demonstrated.')

## 10 · Compile PDF Report + Zip Artifact

Combines all generated figures into a single PDF (no LaTeX required) and
packages trace CSVs + individual figures into a zip for download.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import datetime

REPORT_PDF = COLAB_OUT / 'dsfb_real_data_report.pdf'

def _cover_page():
    fig = plt.figure(figsize=(8.27, 11.69))
    ax  = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
    ax.add_patch(mpatches.FancyBboxPatch((0.05,0.80), 0.90, 0.14,
                                          boxstyle='round,pad=0.01',
                                          facecolor='#2166ac', edgecolor='none'))
    ax.text(0.50, 0.89, 'DSFB Oil & Gas', ha='center', va='center',
            fontsize=24, fontweight='bold', color='white')
    ax.text(0.50, 0.83, 'Real-World Empirical Figures', ha='center', va='center',
            fontsize=14, color='white')
    ax.text(0.50, 0.74,
            'Drift-Slew Fusion Bootstrap\nStructural Residual Semantics',
            ha='center', va='center', fontsize=11, color='#222', style='italic')
    ax.text(0.50, 0.62,
            'Data provenance:\n'
            '  Petrobras 3W v2.0.0  [CC BY 4.0]          9 087 steps\n'
            '  Equinor Volve 15/9-F-15  [Volve DLV1.0]   5 326 steps\n'
            '  RPDBCS ESPset  [MIT License]               6 032 steps',
            ha='center', va='center', fontsize=9, color='#333',
            bbox=dict(boxstyle='round', facecolor='#f5f5f5', edgecolor='#ccc', pad=0.8))
    ax.text(0.50, 0.47,
            'DSFB is a read-only grammar observer.\n'
            'It does not predict failures, detect anomalies, or control systems.\n'
            'Fault labels appear only as post-hoc annotation metadata.\n'
            'DSFB augments existing probabilistic methods -- it does not replace them.',
            ha='center', va='center', fontsize=9, color='#222',
            bbox=dict(boxstyle='round', facecolor='#fff3e0', edgecolor='#e65100', pad=0.8))
    ax.text(0.50, 0.05,
            f'Generated {datetime.date.today().isoformat()} '
            'by cargo run --example export_grammar_traces',
            ha='center', va='center', fontsize=7, color='#888')
    return fig

print(f'Compiling PDF -> {REPORT_PDF.name} ...')
saved_pngs = sorted(COLAB_OUT.glob('fig*.png'))

with PdfPages(str(REPORT_PDF)) as pdf:
    cover = _cover_page()
    pdf.savefig(cover, bbox_inches='tight')
    plt.close(cover)
    for png_p in saved_pngs:
        img_data = plt.imread(str(png_p))
        fig2, ax2 = plt.subplots(figsize=(11, 8.5))
        ax2.imshow(img_data); ax2.axis('off')
        ax2.set_title(png_p.stem, fontsize=7, pad=3)
        pdf.savefig(fig2, bbox_inches='tight')
        plt.close(fig2)
    d = pdf.infodict()
    d['Title']   = 'DSFB Oil & Gas -- Real-World Empirical Figures'
    d['Subject'] = 'Petrobras 3W | Equinor Volve | RPDBCS ESPset'

size_kb = REPORT_PDF.stat().st_size // 1024
print(f'  Report: {REPORT_PDF.name}  ({size_kb} KB, {1+len(saved_pngs)} pages)')

In [ ]:
import zipfile

ZIP_PATH = COLAB_OUT / 'dsfb_real_data_figures.zip'

with zipfile.ZipFile(str(ZIP_PATH), 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pdf_f in sorted(FIGURES_DIR.glob('fig_*.pdf')):
        zf.write(str(pdf_f), f'figures/{pdf_f.name}')
    for pdf_f in sorted(COLAB_OUT.glob('fig*.pdf')):
        zf.write(str(pdf_f), f'colab_output/{pdf_f.name}')
    zf.write(str(REPORT_PDF), 'dsfb_real_data_report.pdf')
    for csv_f in sorted(TRACE_DIR.glob('*.csv')):
        zf.write(str(csv_f), f'trace_data/{csv_f.name}')
    booklet = FIGURES_DIR / 'all_figures.pdf'
    if booklet.exists():
        zf.write(str(booklet), 'all_figures.pdf')

zip_kb = ZIP_PATH.stat().st_size // 1024
print(f'Zip: {ZIP_PATH.name}  ({zip_kb} KB)')
with zipfile.ZipFile(str(ZIP_PATH)) as zf:
    for info in zf.infolist():
        print(f'  {info.filename:<55} {info.file_size//1024:>5} KB')

In [ ]:
if IN_COLAB:
    from google.colab import files
    print(f'Downloading {ZIP_PATH.name} ...')
    files.download(str(ZIP_PATH))
    files.download(str(REPORT_PDF))
else:
    print('Local mode. Artifacts at:')
    print(f'  Report PDF : {REPORT_PDF}')
    print(f'  Zip        : {ZIP_PATH}')

## 11 · Reproducibility Manifest & Scope Boundaries

| Step | Tool | Determinism |
|------|------|-------------|
| Rust toolchain | rustup stable | pinned by Cargo.lock |
| Crate build | cargo build --release | deterministic |
| Grammar engine | export_grammar_traces | provably deterministic (DSFB Thm. 1) |
| Trace CSVs | Rust example | floats at 6 s.f. |
| Figures | matplotlib PdfPages | reproducible with same trace data |

The DSFB grammar automaton is provably deterministic: identical residual input
yields identical token output regardless of execution environment or run order.
See paper §IV.

---

### Final Scope Boundaries

1. **Single-channel.** All results cover one channel per dataset.  Multi-channel,
   cross-well, and cross-asset generalisation are **not demonstrated**.

2. **Envelope calibration on evaluation data.** Independent test-set calibration
   is required before any field-deployment claim.

3. **No detection claim.** DSFB emits grammar tokens; it does not detect faults,
   predict failures, or classify anomalies.

4. **ESPset corpus structure.** RPDBCS ESPset is a laboratory snapshot corpus.
   Field NCR for continuous ESP telemetry is expected to be substantially higher.

5. **TRL 3 only.** Phase II field validation required before production use.